# In Class Activity April 14th, 2026

In [ ]:
# pip install optuna

### Importing libraries, preparing data, initial EDA

In [12]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, RepeatedStratifiedKFold
from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


In [2]:
# importing data
adult = pd.read_csv('adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [3]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





- Data set has an unbalanced target variable (25/75)
    - binary target var
- Majority Men (66/33)
- 1996 data
- need to encode target var to 1 and 0 
- replace "?" with NaN
- there are two education predictors, need to remove one


I will use **F-1 score**, because both resopnses matter and the target variable is unbalanced and binary


### Data Preprocessing (minimal) and Baseline Model

In [4]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [5]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [6]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The baseline score was 0.712. I plan on improving this F-1 score by incoperating Optuna Tuning. I will have Optuna maximize F-1 score, include "imbalance" parameters in the search space (scale_pos_weight), explore using min_child_weight to prevent model overfit.

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [ ]:
# 3 setups: imbalance (scale_pos_weight), tree size (max_depth), training speed (learning_rate + n_estimators)
spw = (y == 0).sum() / (y == 1).sum()
setups = [
    ("scale_pos_weight", {"scale_pos_weight": spw}),
    ("max_depth=3", {"max_depth": 3}),
    ("learning_rate", {"learning_rate": 0.05, "n_estimators": 300}),
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_f1, best_label = -1.0, ""

for label, extra in setups:
    model = XGBClassifier(random_state=42, enable_categorical=True, **extra)
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1")
    mean_f1 = scores.mean()
    print(label, "mean F1", round(mean_f1, 4), "std", round(scores.std(), 4))
    if mean_f1 > best_f1:
        best_f1, best_label = mean_f1, label

print("best:", best_label, "F1", round(best_f1, 4))


scale_pos_weight mean F1 0.7146 std 0.0047
max_depth=3 mean F1 0.7122 std 0.0073
learning_rate mean F1 0.7126 std 0.0051
best: scale_pos_weight F1 0.7146


Exception ignored in: <function ResourceTracker.__del__ at 0x10276dbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1032e9bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1031a9bc0>
Traceback (most recent call last

### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [17]:
spw_train = (y_train == 0).sum() / (y_train == 1).sum()

param_grid = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [100, 200],
    "scale_pos_weight": [1.0, spw_train],
    "subsample": [0.8, 1.0],
}

grid_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    XGBClassifier(random_state=42, enable_categorical=True),
    param_grid,
    cv=grid_cv,
    scoring="f1",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print("best params:", grid_search.best_params_)
print("best CV F1:", round(grid_search.best_score_, 4))

xgb_grid_best = grid_search.best_estimator_
y_pred_grid = xgb_grid_best.predict(X_test)
print("\ntest set:")
print(classification_report(y_test, y_pred_grid))


best params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'scale_pos_weight': np.float64(3.1793774735265803), 'subsample': 0.8}
best CV F1: 0.7168

test set:
              precision    recall  f1-score   support

           0       0.94      0.84      0.89      7431
           1       0.63      0.84      0.72      2338

    accuracy                           0.84      9769
   macro avg       0.78      0.84      0.80      9769
weighted avg       0.87      0.84      0.85      9769



### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [18]:
param_dist = {
    "scale_pos_weight": np.linspace(1.0, 10.0, num=10),
    "max_depth": np.arange(3, 11),
    "learning_rate": np.linspace(0.01, 0.3, num=10),
    "n_estimators": np.arange(100, 401, 50),
    "subsample": np.linspace(0.7, 1.0, num=7),
}

xgb_random = RandomizedSearchCV(
    XGBClassifier(random_state=42, enable_categorical=True),
    param_distributions=param_dist,
    n_iter=20,
    cv=skf,
    scoring="f1",
    random_state=42,
    n_jobs=-1,
)
xgb_random.fit(X_train, y_train)

print("Best parameters from RandomizedSearchCV:", xgb_random.best_params_)
print("Best CV F1:", round(xgb_random.best_score_, 4))

xgb_random_best = xgb_random.best_estimator_
y_pred_random = xgb_random_best.predict(X_test)
print("\nClassification report (test set):")
print(classification_report(y_test, y_pred_random))


Best parameters from RandomizedSearchCV: {'subsample': np.float64(0.7), 'scale_pos_weight': np.float64(3.0), 'n_estimators': np.int64(100), 'max_depth': np.int64(10), 'learning_rate': np.float64(0.07444444444444444)}
Best CV F1: 0.7163

Classification report (test set):
              precision    recall  f1-score   support

           0       0.94      0.85      0.89      7431
           1       0.63      0.83      0.72      2338

    accuracy                           0.84      9769
   macro avg       0.79      0.84      0.80      9769
weighted avg       0.87      0.84      0.85      9769



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [19]:
def objective(trial):
    params = {
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 10.0),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
    }
    model = XGBClassifier(random_state=42, enable_categorical=True, **params)
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring="f1")
    return scores.mean()


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=20, show_progress_bar=True)

print("Best parameters from Optuna:", study.best_params)
print("Best CV F1:", round(study.best_value, 4))

xgb_optuna_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    **study.best_params,
)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print("\nClassification report (test set):")
print(classification_report(y_test, y_pred_optuna))


[I 2026-04-15 09:35:44,577] A new study created in memory with name: no-name-6ea520e8-be5d-426a-82f6-0bdf5df27a54


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-15 09:35:51,133] Trial 0 finished with value: 0.6897724225338048 and parameters: {'scale_pos_weight': 4.370861069626263, 'max_depth': 10, 'learning_rate': 0.22227824312530747, 'n_estimators': 300, 'subsample': 0.7468055921327309}. Best is trial 0 with value: 0.6897724225338048.
[I 2026-04-15 09:35:52,879] Trial 1 finished with value: 0.7219499894678083 and parameters: {'scale_pos_weight': 2.403950683025824, 'max_depth': 3, 'learning_rate': 0.2611910822747312, 'n_estimators': 300, 'subsample': 0.9124217733388136}. Best is trial 1 with value: 0.7219499894678083.
[I 2026-04-15 09:35:56,284] Trial 2 finished with value: 0.6869080934715063 and parameters: {'scale_pos_weight': 1.185260448662222, 'max_depth': 10, 'learning_rate': 0.2514083658321223, 'n_estimators': 150, 'subsample': 0.7545474901621302}. Best is trial 1 with value: 0.7219499894678083.
[I 2026-04-15 09:35:59,529] Trial 3 finished with value: 0.7150970778140163 and parameters: {'scale_pos_weight': 2.650640588680904, '

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


The optuna mthod produced the best results.I prefer OPtuna, becasue it is not random and it leanrs from previous param combinations. The model is adaptive and more efficient, especially with a larger search area to deal with. 